In [16]:
# kernel  3.10.6

import pandas as pd
import numpy as np
import torch
import torch_directml
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import pacmap
from sklearn.cluster import HDBSCAN


# import umap

# Open file and start cleaning

In [2]:
file = 'conicet_dataset.csv'

df = pd.read_csv(file)

df.sample(10)


,oai_identifier,title,creator,subject,description,filiation,date,type,identifier,language,rights,format,relation,publisher,source,contributor,coverage
29163,oai:ri.conicet.gov.ar:11336/143868,Impact of lipid sources on quality traits of m...,"Ramella, Alberto|Roda, Gabriella|Pavlovic, Rad...",CANNABINOIDS|CANNABIS OILS|HS-SPME-GC/MS|LCHRM...,The feasibility of the use of two lipid source...,"Fil: Ramella, Alberto. Farmacia Dott.ri Giulia...",2020-07,info:eu-repo/semantics/article|info:ar-repo/se...,"http://hdl.handle.net/11336/143868|Ramella, Al...",eng,info:eu-repo/semantics/openAccess|https://crea...,application/pdf|application/pdf,info:eu-repo/semantics/altIdentifier/url/https...,Molecular Diversity Preservation International,NaN,NaN,NaN
149912,oai:ri.conicet.gov.ar:11336/73979,La teoría marxista de la dependencia desde una...,"Costantino, María Agostina|Laterra, Patricia",TEORÍA DE LA DEPENDENCIA|ECONOMÍA FEMINISTA|DE...,La teoría marxista de la dependencia surgió en...,"Fil: Costantino, María Agostina. Consejo Nacio...",2017-12,info:eu-repo/semantics/article|info:ar-repo/se...,"http://hdl.handle.net/11336/73979|Costantino, ...",spa,info:eu-repo/semantics/openAccess|https://crea...,application/pdf|application/pdf,info:eu-repo/semantics/altIdentifier/url/https...,Universidad pública en Campinhas,NaN,NaN,NaN
110074,oai:ri.conicet.gov.ar:11336/273564,Hogar y salud: medicina casera adventista en l...,"Rivero, María Dolores",SALUD|HOGAR|MANUAL|ADVENTISMO|https://purl.org...,El presente trabajo coloca en perspectiva anal...,"Fil: Rivero, María Dolores. Consejo Nacional d...",2025-05,info:eu-repo/semantics/article|info:ar-repo/se...,"http://hdl.handle.net/11336/273564|Rivero, Mar...",spa,info:eu-repo/semantics/openAccess|https://crea...,application/pdf|application/pdf,info:eu-repo/semantics/altIdentifier/url/https...,Centro de Estudios e Investigaciones Laborales,NaN,NaN,NaN
65915,oai:ri.conicet.gov.ar:11336/201390,PD-1 and LAG-3 expression in EBV-associated pe...,"Jimenez, Oscar Eduardo|Mangiaterra, Tamara Sol...",HODGKIN LYMPHOMA|LAG-3|PD-1|PEDIATRIC|TOLEROGE...,"In pediatric Hodgkin lymphoma (HL), the inabil...","Fil: Jimenez, Oscar Eduardo. Gobierno de la Ci...",2022-08,info:eu-repo/semantics/article|info:ar-repo/se...,"http://hdl.handle.net/11336/201390|Jimenez, Os...",eng,info:eu-repo/semantics/openAccess|https://crea...,application/pdf|application/pdf|application/pd...,info:eu-repo/semantics/altIdentifier/url/https...,Frontiers Media,NaN,NaN,NaN
1725,oai:ri.conicet.gov.ar:11336/102556,Genetic inhibition of calcineurin induces dias...,"Gelpi, Ricardo Jorge|Gao, Shumin|Zhai, Peiyong...",hypertrophy|Diastole|Hemodynamics|https://purl...,Genetic inhibition of calcineurin induces dias...,"Fil: Gelpi, Ricardo Jorge. Consejo Nacional de...",2009-11,info:eu-repo/semantics/article|info:ar-repo/se...,"http://hdl.handle.net/11336/102556|Gelpi, Rica...",eng,info:eu-repo/semantics/openAccess|https://crea...,application/pdf|application/pdf|application/pdf,info:eu-repo/semantics/altIdentifier/doi/10.11...,American Physiological Society,NaN,NaN,NaN
41242,oai:ri.conicet.gov.ar:11336/161396,Chemical characterization free radical scaveng...,"Gómez, J.|Simirgiotis, M. J.|Lima, Beatriz Viv...",CHEMICAL CHARACTERIZATION|FREE RADICAL SCAVENG...,"The resins are nonvolatile products of pants, ...","Fil: Gómez, J.. Universidad Nacional de San Ju...",2019,info:eu-repo/semantics/publishedVersion|info:e...,http://hdl.handle.net/11336/161396|Chemical ch...,eng,info:eu-repo/semantics/openAccess|https://crea...,application/pdf|application/pdf|application/pdf,info:eu-repo/semantics/altIdentifier/url/https...,Sociedad Biológica de Cuyo,NaN,NaN,Nacional
153951,oai:ri.conicet.gov.ar:11336/79850,Efecto Kondo en moléculas y átomos superficies...,"Fernández, Joaquín",Efecto Kondo|Moléculas|Átomos|Superficies Metá...,En este trabajo se presenta un estudio detalla...,"Fil: Fernández, Joaquín. Consejo Nacional de I...",2019-03-21,info:eu-repo/semantics/doctoralThe

In [11]:
df['language'].value_counts()

language
eng    487499
spa    169703
Name: count, dtype: int64

In [3]:
# erase uneeded columns

columns_to_keep = ['title', 'description', 'filiation', 'language']
df.drop(columns=[col for col in df.columns if col not in columns_to_keep], inplace=True)

#retain only papers in spanish or english
df = df[df['language'].isin(['spa', 'eng'])]

In [4]:
df['language'].value_counts()

language
eng    89721
spa    72918
Name: count, dtype: int64

In [5]:
# create a new column joining the texts of title and abstracts
df['text'] = df['title'].fillna('') + " " + df['description'].fillna('')


# Calculate embeddings

In [ ]:
# Using my AMD hardware
device = torch_directml.device()
print(f"Usando el dispositivo: {device}")

# 1. Cargar el modelo
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')

# --- PARCHE PARA EL TENSOR DE XLM-ROBERTA ---
inner_model = model._first_module().auto_model
if hasattr(inner_model.embeddings, "token_type_ids"):
    delattr(inner_model.embeddings, "token_type_ids")
# --------------------------------------------

model = model.to(device)
model.eval()

Usando el dispositivo: privateuseone:0


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
)

In [ ]:
# 2. Preparar los datos
texts = df['text'].tolist()
batch_size = 32 
paper_embeddings = []

print("Calculando embeddings manualmente en AMD GPU...")

# 3. BUCLE MANUAL DE INFERENCIA
with torch.no_grad():
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i : i + batch_size]
        
        # A. Usamos el tokenizador interno indicando que devuelva tensores de PyTorch ('pt')
        features = model.tokenizer(
            batch_texts, 
            padding=True, 
            truncation=True, 
            return_tensors="pt"
        )
        
        # B. Movemos a la GPU filtrando por seguridad (solo tensores)
        features = {k: v.to(device) for k, v in features.items() if isinstance(v, torch.Tensor)}
        
        # C. Pasar por el modelo
        out_features = model(features)
        
        # D. Extraer los embeddings finales
        embeddings = out_features['sentence_embedding'].cpu().numpy()
        paper_embeddings.extend(embeddings)

# 4. Guardar resultados
df['emb'] = paper_embeddings
print("¡Embeddings calculados con éxito!")

Calculando embeddings manualmente en AMD GPU...


100%|██████████| 5083/5083 [1:25:05<00:00,  1.00s/it]

¡Embeddings calculados con éxito!


In [3]:
df['emb']

0         [0.0396710001, 0.0818290636, -0.00394671597, 0...
1         [-0.0608286262, 0.241804481, -0.0126218796, -0...
2         [-0.108535483, 0.369745821, -0.0108462442, -0....
3         [-0.00273833517, 0.0615373924, -0.0205853097, ...
4         [-0.07914443, 0.04663156, 0.00068415, -0.02516...
                                ...                        
162634    [0.105090693, 0.133802608, -0.0103731481, 0.03...
162635    [-0.0353654772, 0.301036268, -0.00970180705, -...
162636    [0.0684111714, 0.198618144, -0.00100525399, 0....
162637    [-0.0755715519, 0.0403095335, -0.00459716003, ...
162638    [-0.0731734335, 0.124165833, -0.0176578052, -0...
Name: emb, Length: 162639, dtype: object

In [11]:
df.columns

Index(['title', 'description', 'filiation', 'language', 'text', 'emb', 'x',
       'y'],
      dtype='object')

# Save to csv

In [ ]:
df.to_csv(r'C:\Users\Usuario\Desktop\conicet_emb_umap_2026-09-02.csv', index=False)